# Demo orchestrator

In [1]:
import sys
import os
import random
import torch
import pandas as pd
import logging

# 1. Setup path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 3. Import Pipeline components
from peptide_pipeline.orchestrator.orchestrator import Orchestrator
from peptide_pipeline.chemist.agent_v1 import ChemistAgent
from peptide_pipeline.chemist.agent_v1.config_chemist import ChemistConfig
from peptide_pipeline.biologist.esm_biologist_zscore import ESMBiologistZscore
from peptide_pipeline.generator.cvae_generator import CVAEGenerator


/home/niels/pyvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. Prepare data and train the CVAE generator used by the orchestrator
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader

json_path = os.path.join(project_root, "ai_training_peptides.json")
if not os.path.exists(json_path):
    raise FileNotFoundError(f"Training file not found: {json_path}")

json_loader = JSONDataLoader()
json_loader.load_data(
    json_path,
    columns=["sequence", "length", "cathionicity", "hydrophobicity"],
)
data = json_loader.get_data()

MAX_LEN = 14
MIN_LEN = 5
filtered_data = data[(data["length"] >= MIN_LEN) & (data["length"] <= MAX_LEN)].copy()
if filtered_data.empty:
    raise ValueError("No peptides available in the requested length range.")

print(f"Loaded peptides: {len(data)} | Filtered ({MIN_LEN}-{MAX_LEN} aa): {len(filtered_data)}")

# Build constraints used to produce conditioning tensors during training
min_max_values = {}
for prop in ["length", "cathionicity", "hydrophobicity"]:
    min_max_values[prop] = {
        "min": float(filtered_data[prop].min()),
        "max": float(filtered_data[prop].max()),
    }

constraints_default = {
    "size": {"min": min_max_values["length"]["min"], "max": min_max_values["length"]["max"]},
    "molecular_weight": {"min": 500, "max": 3000},
    "net_charge_pH5_5": {"min": -5, "max": 5},
    "isoelectric_point": {"min": 3, "max": 11},
    "hydrophobicity": {"min": min_max_values["hydrophobicity"]["min"], "max": min_max_values["hydrophobicity"]["max"]},
    "cathionicity": {"min": min_max_values["cathionicity"]["min"], "max": min_max_values["cathionicity"]["max"]},
    "hydrophobic_moment": {"min": 0, "max": 1},
    "logp": {"min": -3, "max": 3},
    "boman_index": {"min": -5, "max": 5},
    "h_bond_donors": {"min": 0, "max": 10},
    "h_bond_acceptors": {"min": 0, "max": 10},
    "tpsa": {"min": 0, "max": 200},
}

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
PAD_IDX = 20
VOCAB_SIZE = 21

def encode_with_pad(seqs, max_len):
    x = torch.zeros(len(seqs), max_len * VOCAB_SIZE)
    for i, seq in enumerate(seqs):
        for j in range(max_len):
            x[i, j * VOCAB_SIZE + PAD_IDX] = 1.0
        for j, aa in enumerate(seq[:max_len]):
            if aa in AA_INDEX:
                x[i, j * VOCAB_SIZE + PAD_IDX] = 0.0
                x[i, j * VOCAB_SIZE + AA_INDEX[aa]] = 1.0
    return x

sequences = [s for s in filtered_data["sequence"].tolist() if MIN_LEN <= len(s) <= MAX_LEN]
lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
x = encode_with_pad(sequences, MAX_LEN)

generator = CVAEGenerator(
    max_len=MAX_LEN,
    latent_dim=16,
    hidden_dim=64,
    condition_dim=32,
)

conditions = generator._constraints_to_condition_tensor(
    constraints=constraints_default,
    count=len(sequences),
    device=generator.device,
 )
x = x.to(generator.device)
lengths = lengths.to(generator.device)

EPOCHS = 100
BATCH_SIZE = 64
LR = 1e-3

print(
    f"Training CVAE on {len(sequences)} sequences | device={generator.device} | "
    f"epochs={EPOCHS}, batch_size={BATCH_SIZE}"
)
generator.train_model(
    data=x,
    conditions=conditions,
    lengths=lengths,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    kl_anneal_epochs=EPOCHS,
)

print("CVAE ready. Variable 'generator' can now be used by the orchestrator.")

Loaded peptides: 3646 | Filtered (5-14 aa): 3203
Training CVAE on 3203 sequences | device=cuda | epochs=100, batch_size=64
  Epoch 050/100 | Recon: 2.4522 | KL: 0.0000 | KL weight: 0.49
  Epoch 100/100 | Recon: 2.4568 | KL: 0.0000 | KL weight: 0.99
CVAE ready. Variable 'generator' can now be used by the orchestrator.


## 2. Init all agents + orchestrator

In [3]:
chemist_config = ChemistConfig() #defaut config ph 7 
chemist = ChemistAgent(config = chemist_config)
biologist = ESMBiologistZscore(
    reference_peptide="RLLKRF", 
    model_name="facebook/esm2_t6_8M_UR50D"
)

orchestrator = Orchestrator(generator, chemist, biologist)
print("Orchestrator Initialized.")

Loading weights: 100%|██████████| 107/107 [00:00<00:00, 1613.03it/s]
EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Orchestrator Initialized.


## 3. Exécution de la Pipeline

Nous partons du peptide **"RLLKRF"**. L'orchestrateur va :
1. Demander au VAE de générer des variantes (mutation via l'espace latent).
2. Filtrer chimiquement.
3. Scorer biologiquement.
4. Répéter le processus sur les survivants.

In [4]:
results = orchestrator.run(
    nb_iterations=5,
    nb_peptides=50,
    top_k=5,
    exploration_rate=0.3, # Bruit injecté dans l'espace latent pour créer des variants
    initial_peptide="RLLKRF"
)

print(f"\nPipeline Finished. {len(results)} candidates found.")

Sequence GLKRLKVI is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence PRPRV is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence GWAWKLVV is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence RRKLHFKN is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.
Sequence KKKRKWKAK is invalid. The sequence contains non-standard amino acids. Only the 20 standard amino acids are allowed.



Pipeline Finished. 1 candidates found.


## 4. Résultats

In [5]:
df_results = []
for res in results:
    row = {
        "Peptide": res.get("peptide"),
        "Score ESM": res.get("score"),
        "Score Chem": res.get("chemist_score"),
        "In limits": res.get("in_limits"),
    }
    row.update(res.get("properties", {}))
    df_results.append(row)

df = pd.DataFrame(df_results)
if not df.empty:
    preferred_cols = [
        "Peptide",
        "Score ESM",
        "Score Chem",
        "In limits",
        "molecular_weight",
        "logp",
    ]
    cols = [c for c in preferred_cols if c in df.columns]
    cols += [c for c in df.columns if c not in cols]
    display(df[cols])
else:
    print("Aucun résultat.")

,Peptide,Score ESM,Score Chem,In limits
0,KKKRKWKAK,0.245312,1.0,True
